In [ ]:

import pandas as pd
from pathlib import Path

# ==========================================
# 1. CONFIGURATION
# ==========================================

INPUT_FILE = Path(
    "output/second_reports/tanggal_enriched_cleaned.xlsx"
)

OUTPUT_DIR = Path("output/second_reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "attendance_feature_engineered.xlsx"


# ==========================================
# 2. LOAD DATA
# ==========================================

df = pd.read_excel(INPUT_FILE)

print("Dataset loaded successfully!")
print("Total records:", len(df))


# ==========================================
# 3. CREATE CLOCK TIME LABELS
# ==========================================

clock_columns = {
    "clock_in_time": "clock_in_time_label",
    "clock_out_time": "clock_out_time_label",
    "clock_in_time_be": "clock_in_time_be_label",
    "clock_out_time_be": "clock_out_time_be_label"
}


# Validate required columns

for original_col in clock_columns.keys():

    if original_col not in df.columns:
        raise ValueError(
            f"Column not found: {original_col}"
        )


# Check whether clock values exist

def check_clock_value(series):

    value = series.astype("string").str.strip()

    has_value = (
        series.notna()
        & value.ne("").fillna(False)
        & ~value.str.upper().isin(
            ["NULL", "NONE", "NAN", "NAT", "-"]
        ).fillna(False)
    )

    return has_value.map({
        True: "Available",
        False: "Missing"
    })


# Generate four individual clock labels

for original_col, new_col in clock_columns.items():

    df[new_col] = check_clock_value(
        df[original_col]
    )


# ==========================================
# 4. CREATE COMBINED CLOCK LABELS
# ==========================================

# Clock-in:
# Available if original OR BE has a value

df["clock_in_label"] = (
    df["clock_in_time_label"].eq("Available")
    |
    df["clock_in_time_be_label"].eq("Available")
).map({
    True: "Available",
    False: "Missing"
})


# Clock-out:
# Available if original OR BE has a value

df["clock_out_label"] = (
    df["clock_out_time_label"].eq("Available")
    |
    df["clock_out_time_be_label"].eq("Available")
).map({
    True: "Available",
    False: "Missing"
})


# ==========================================
# 5. CREATE RECORD COMPLETENESS
# ==========================================

# Complete:
# Both clock-in and clock-out exist

# Incomplete:
# Only one of them exists

# Missing:
# Neither exists

conditions = [
    (
        df["clock_in_label"].eq("Available")
        &
        df["clock_out_label"].eq("Available")
    ),

    (
        df["clock_in_label"].eq("Available")
        |
        df["clock_out_label"].eq("Available")
    )
]

choices = [
    "Complete",
    "Incomplete"
]

df["record_completeness"] = pd.Series(
    "Missing",
    index=df.index,
    dtype="string"
)

df.loc[
    conditions[0],
    "record_completeness"
] = choices[0]

df.loc[
    conditions[1] & ~conditions[0],
    "record_completeness"
] = choices[1]


# ==========================================
# 6. CREATE STORE TYPE
# ==========================================

if "location_name" not in df.columns:
    raise ValueError(
        "location_name column not found"
    )

location = (
    df["location_name"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# Location classification rules

store_conditions = [
    location.str.contains("HYFRESH", na=False),
    location.str.contains("HEAD OFFICE", na=False),
    location.str.contains("DISTRIBUTION CENTER", na=False),
    location.str.contains("HYPERMART", na=False),
    location.str.contains("DC", na=False),
    location.str.contains("FMX", na=False),
    location.str.contains("SSO", na=False),
    location.str.contains("BOSTON", na=False),
    location.str.contains("B&B", na=False),
    location.str.contains("TOWAKI", na=False),
    location.str.contains("FOODMART", na=False)
]

store_choices = [
    "HFR",
    "HO",
    "DC",
    "HPRM",
    "DC",
    "FMX",
    "SSO",
    "B&B",
    "B&B",
    "TT",
    "FDM"
]


# Initialize store_type

df["store_type"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)


# Assign store types

for condition, choice in zip(
    store_conditions,
    store_choices
):

    df.loc[
        condition & df["store_type"].isna(),
        "store_type"
    ] = choice


# Unclassified locations

df["store_type"] = df["store_type"].fillna("OTHER")


# ==========================================
# 7. EXPORT FEATURE-ENGINEERED DATA
# ==========================================

df.to_excel(
    OUTPUT_FILE,
    index=False
)


# ==========================================
# 8. SUMMARY REPORT
# ==========================================

print("\n==========================================")
print("FEATURE ENGINEERING COMPLETED")
print("==========================================")

print("\nTotal records:", len(df))


# Store type distribution

print("\nSTORE TYPE DISTRIBUTION")
print(
    df["store_type"].value_counts(dropna=False)
)


# Individual clock labels

print("\nINDIVIDUAL CLOCK LABELS")

for col in clock_columns.values():

    print(f"\n{col}")

    print(
        df[col].value_counts(dropna=False)
    )


# Combined clock labels

print("\nCOMBINED CLOCK LABELS")

for col in [
    "clock_in_label",
    "clock_out_label"
]:

    print(f"\n{col}")

    print(
        df[col].value_counts(dropna=False)
    )


# Record completeness

print("\nRECORD COMPLETENESS")

print(
    df["record_completeness"].value_counts(
        dropna=False
    )
)


# Export location

print("\nOutput:", OUTPUT_FILE)


Dataset loaded successfully!
Total records: 27711

FEATURE ENGINEERING COMPLETED

Total records: 27711

STORE TYPE DISTRIBUTION
store_type
HPRM    19493
DC       2419
FDM      2173
HO       1665
HFR      1345
B&B       472
FMX       113
SSO        25
TT          6
Name: count, dtype: Int64

INDIVIDUAL CLOCK LABELS

clock_in_time_label
clock_in_time_label
Available    26975
Missing        736
Name: count, dtype: int64

clock_out_time_label
clock_out_time_label
Available    23852
Missing       3859
Name: count, dtype: int64

clock_in_time_be_label
clock_in_time_be_label
Available    26562
Missing       1149
Name: count, dtype: int64

clock_out_time_be_label
clock_out_time_be_label
Available    23256
Missing       4455
Name: count, dtype: int64

COMBINED CLOCK LABELS

clock_in_label
clock_in_label
Available    26975
Missing        736
Name: count, dtype: int64

clock_out_label
clock_out_label
Available    23852
Missing       3859
Name: count, dtype: int64

RECORD COMPLETENESS
record_compl